# Единая методика финансового эффекта

Минимальный генератор отчёта `data/fin_effect_report.html`.

Все формулы, допущения, диагностика качества, ITT, compliance-сценарии,
чувствительность и сезонная экстраполяция находятся в HTML.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_here, *_here.parents) if (path / "pyproject.toml").exists()
)
for path in (PROJECT_ROOT / "src", PROJECT_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from querulus.fin_effect.excel_monitoring import (
    RETRO_AS_OF_DEFAULT,
    VITRINA_TABLE_DEFAULT,
    estimate_monitoring_effect,
    load_monitoring_frame,
)
from querulus.fin_effect.monitoring_report import (
    REPORT_FILENAME,
    write_error_html,
    write_monitoring_html,
)

DATA_DIR = PROJECT_ROOT / "monitoring" / "fin_effects" / "data"
REPORT_PATH = DATA_DIR / REPORT_FILENAME
RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
LOOKBACK_YEARS = 2.0
RETRO_AS_OF = RETRO_AS_OF_DEFAULT
T_CALC = None
DISCOUNT_RATE = 0.12
RESIDUAL_SHARE = 0.07
BOOTSTRAP_ITERATIONS = 200

In [ ]:
try:
    monitoring_df = load_monitoring_frame(
        source="mssql",
        table=VITRINA_TABLE_DEFAULT,
    )
    if not RETRO_PARQUET.exists():
        raise FileNotFoundError(
            f"Не найден финальный ретро-датасет: {RETRO_PARQUET}"
        )
    retro_df = pd.read_parquet(RETRO_PARQUET)
    result = estimate_monitoring_effect(
        monitoring_df,
        retro_df,
        t_calc=T_CALC,
        residual_share=RESIDUAL_SHARE,
        discount_rate=DISCOUNT_RATE,
        lookback_years=LOOKBACK_YEARS,
        retro_as_of=RETRO_AS_OF,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    )
    report_path = write_monitoring_html(
        result,
        REPORT_PATH,
        source_label=VITRINA_TABLE_DEFAULT,
    )
except Exception as exc:
    report_path = write_error_html(
        exc,
        REPORT_PATH,
        source_label=VITRINA_TABLE_DEFAULT,
    )
    raise

print(f"HTML → {report_path}")